In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# configure a simple exporter for the telemetry traces.
import os

import logfire

logfire.configure(
    token=os.environ["LOGFIRE_TOKEN"],
    service_name="ai-assist-demo",
    scrubbing=False,
    send_to_logfire=True,
)

Let's walk through a small but realistic example. 

This notebook tells a story:
1. We have a task that we want to solve
2. We gather example data for the problem
3. We define the program (function) that uses an LM to solve that problem
4. We choose the LM and prompt that we want to test against the task
5. We define some metrics (programmatic or LLM as a judge) that measure how well or bad our program + LM + template solve the problem
6. We run the experiment: execute the program for all the examples and score all the metrics
7. Finally, we have an LM optimizer run the experiment, change the prompt template, and keep iterating to try and improve the metric scores over the examples

## Dataset
The task is **product review sentiment analysis**: given a customer review, predict whether the sentiment is `positive`, `neutral`, or `negative`, and produce a short justification for the call.

Sentiment labels are always the English strings `positive`, `neutral`, or `negative` — even when the review itself is in another language.

We start with the **partial** dataset (`data/partial.jsonl`): nine labelled reviews, mostly English plus one each in Spanish, German, French, and Italian. The `reference` field is the ground-truth label. Later we switch to `data/full.jsonl` for the optimization section.


In [3]:
from promptuna.projects import resolve_examples, resolve_project_dir

project_dir = resolve_project_dir("classify_sentiment")
examples = resolve_examples(project_dir, "partial")

Logfire project URL: https://logfire-eu.pydantic.dev/nachollorca/ai-assist-demo

## Target Function
Now we define the program — the **thing we actually want to evaluate**.

> Note that it is not just a thin wrapper around `complete()`: each program makes **exactly one** LM completion, wrapped in a deterministic **scaffold** — code before the call (input shaping, template rendering) and after (parsing, coercion, fallbacks). In production, users rarely hit the raw completion; they hit the completion plus its scaffold. The harness evaluates that full product.

The function must adhere to the `Program` protocol: take its **named inputs**, the **prompt template**, and a **model id**, then return whatever the downstream scorers should consume. The harness unpacks `Example.inputs` as keyword arguments, so the parameter names must match the dict keys.

The program lives in `samples/classify_sentiment/programs.py` (`v1`) and is imported below. Keeping it in a `.py` module (rather than defining it in this notebook) lets promptuna introspect the program source when optimizing. Its output schema is declared inside the function body.


In [4]:
from promptuna.projects import resolve_program

classify_sentiment = resolve_program(project_dir, "v1")

## Knobs
The two sweepable axes are the **prompt template** and the **model**. We load `prompts/english.jinja`, which asks the model to write the reason in the same language as the review.

In [5]:
from promptuna.projects import resolve_prompt_template

prompt_template = resolve_prompt_template(project_dir, "english")

model = "mistral:mistral-small-latest"

## Trial
With these ingredients, we can already run a trial: execute the program on one example with the model under test.


In [6]:
from promptuna.run import run_trial

trial = run_trial(
    program=classify_sentiment,
    prompt_template=prompt_template,
    model=model,
    example=examples[0],
)

trial.output

16:23:02.724 chat mistral-small-latest


{'sentiment': 'positive',
 'reason': 'O author expressa satisfação com a duração da bateria, a qualidade da tela e classifica o telefone como o melhor que já possuiu.'}

## Metrics
Now that our trial ran successfully, we can jump into quality measurements.

### Programmatic Metrics
First, a simple deterministic check: does the predicted label match the ground truth? No LLM judge needed — a `ProgrammaticMetric` whose scorer follows the `ProgrammaticScorer` protocol is the right artifact. We declare the value space with an `Ordinal` scale. The full definition lives in `samples/classify_sentiment/metrics.py`.

In [7]:
from promptuna.projects import resolve_metrics

label_correctness, reason_language = resolve_metrics(
    project_dir, ["label_correctness", "reason_language"]
)

Now we score the trial against the metric.

In [8]:
from promptuna.evaluate import score_metric

scoring = score_metric(trial=trial, metric=label_correctness)
scoring.score

Score(raw=True, normalized=1.0, reason="Predicted 'positive' matches reference.")

### LLM as Judge Metrics
The label check is cheap and exact, but it says nothing about whether the justification is written in the same language as the review. The baseline prompt asks for that; we score it with an `LLMJudgeMetric` using the built-in `default_llm_judge` — the judge template is fixed, and the metric description tells it what to check. The full definition lives in `samples/classify_sentiment/metrics.py` as `reason_language`.


Let's see whether the reason matches the review language on our first trial.


In [9]:
language_scoring = score_metric(trial=trial, metric=reason_language)
language_scoring.score

16:23:03.512 chat mistral-medium-latest


Score(raw=False, normalized=0.0, reason='The reason field in the output is written in Portuguese, while the product review in the rendered prompt is in English. Therefore, the output does not meet the metric.')

## Experiment
Finally, we can wire it all together into an experiment: run `classify_sentiment` against every example and score both metrics on each output.

In [10]:
from promptuna.evaluate import evaluate
from promptuna.program import Experiment

experiment = Experiment(
    program=classify_sentiment,
    prompt_template=prompt_template,
    model=model,
)

results = evaluate(
    experiment=experiment,
    examples=examples,
    metrics=[label_correctness, reason_language],
    workers=5,
)

16:23:04.380 chat mistral-small-latest
16:23:04.384 chat mistral-small-latest
16:23:04.387 chat mistral-small-latest
16:23:04.390 chat mistral-small-latest
16:23:04.395 chat mistral-small-latest
16:23:04.914 chat mistral-small-latest
16:23:04.983 chat mistral-small-latest
16:23:05.058 chat mistral-small-latest
16:23:05.102 chat mistral-small-latest
16:23:05.157 chat mistral-medium-latest
16:23:05.532 chat mistral-medium-latest
16:23:05.543 chat mistral-medium-latest
16:23:05.570 chat mistral-medium-latest
16:23:05.897 chat mistral-medium-latest
16:23:06.337 chat mistral-medium-latest
16:23:06.469 chat mistral-medium-latest
16:23:06.887 chat mistral-medium-latest
16:23:06.917 chat mistral-medium-latest


And a rendering utility renders the results.

In [11]:
from IPython.display import Markdown, display

from promptuna.report import render_run

report = render_run(results)
display(Markdown(report))
display(Markdown("---"))

### Quality

| Metric | Score | Replicate noise |
| --- | --- | --- |
| `label_correctness` | 0.89 ± 0.33 (n=9) | 0.00 ± 0.00 (n=9) |
| `reason_language` | 0.56 ± 0.53 (n=9) | 0.00 ± 0.00 (n=9) |

### Reliability

- **Trials**: 9 successful / 9 total
- **Trial failure rate**: 0.00%
- **Scorings**: 18 successful / 18 total
- **Scoring failure rate**: 0.00%

### Error Analysis

#### Example 1 - score 0.50

**Inputs:**

`{'review': 'Stopped charging after three weeks. Support never replied. Avoid.'}`

**Quality:**

- `label_correctness`: 1.00 — Predicted 'negative' matches reference.
- `reason_language`: 0.00 — The reason field in the output is written in Spanish ('El producto falló rápidamente y el servicio de soporte no respondió.'), while the product review in the rendered prompt is in English ('Stopped charging after three weeks. Support never replied. Avoid.'). Thus, the reason is not in the same language as the review.
#### Example 2 - score 0.50

**Inputs:**

`{'review': 'It works. Setup was fine, nothing surprising, nothing to complain about.'}`

**Quality:**

- `label_correctness`: 0.00 — Predicted 'neutral', expected 'positive'.
- `reason_language`: 1.00 — The reason field in the output ('The review is factual and balanced, neither praising nor criticizing the product.') is written in English, which matches the language of the product review in the rendered prompt ('It works. Setup was fine, nothing surprising, nothing to complain about.'). Thus, the metric is satisfied.
#### Example 3 - score 0.50

**Inputs:**

`{'review': '    Camera   is   AMAZING!!!   colors pop, low-light is great.\n\n\nHighly recommend.'}`

**Quality:**

- `label_correctness`: 1.00 — Predicted 'positive' matches reference.
- `reason_language`: 0.00 — The reason field in the output is written in Spanish ('El usuario expresa entusiasmo y recomienda el producto con elogios a sus características.'), while the product review in the rendered prompt is in English ('Camera is AMAZING!!! colors pop, low-light is great. Highly recommend.'). Thus, the metric is not met.

### Telemetry

Totals across successful trials only.

- **Total latency**: 5.732 s
- **Total output tokens**: 315
- **Throughput**: 55.0 tok/s

---

## Optimizer

The harness can also **search for a better prompt template**. `optimize()` evaluates the experiment's current template as a baseline, then repeatedly asks a *proposer* model to rewrite the template from the full trajectory and re-scores each candidate. The archive keeps every checkpoint; `OptimizationResult.best` is the highest-scoring step (not necessarily the last).

In our simple example, the baseline already scores well on the easy reviews in `partial` — and the vague prompt (*"classify the sentiment, the labels are positive / neutral / negative"*) captures everything they need. There's nothing for the optimizer to win. To create some difficulty, we switch to the **`full` dataset**: reviews whose **correct label depends on a labelling rubric the baseline prompt never states**, including harder English cases and more multilingual reviews. The optimizer can only rewrite the prompt template, and it sees the weakest examples plus the scorer reasoning each step — so it can *infer* the rubric from the failures and encode it into a better prompt.

**What success looks like:** the baseline prompt should now score noticeably below 1.0 on these (conflating logistics with the product, getting fooled by sarcasm, scattering the mixed cases, mistaking faint praise or backhanded compliments). A well-optimized prompt — one that spells out "focus on the product, watch for sarcasm/negation, reserve neutral for balanced or factual reviews, and read the overall verdict not just individual adjectives" — should recover most of that gap, though reaching a perfect score is harder now.

In [12]:
from promptuna.projects import resolve_examples

examples = resolve_examples(project_dir, "full")

In [13]:
from promptuna.optimize import optimize

optimization = optimize(
    experiment=experiment,
    examples=examples,
    metrics=[label_correctness, reason_language],
    proposer_model="vertex:gemini-3.5-flash",
    steps=4,
    workers=5,
)

16:23:08.041 chat mistral-small-latest
16:23:08.044 chat mistral-small-latest
16:23:08.049 chat mistral-small-latest
16:23:08.052 chat mistral-small-latest
16:23:08.056 chat mistral-small-latest
16:23:08.574 chat mistral-small-latest
16:23:08.602 chat mistral-small-latest
16:23:08.611 chat mistral-small-latest
16:23:08.649 chat mistral-small-latest
16:23:08.735 chat mistral-small-latest
16:23:09.209 chat mistral-small-latest
16:23:09.226 chat mistral-small-latest
16:23:09.250 chat mistral-small-latest
16:23:09.277 chat mistral-small-latest
16:23:09.341 chat mistral-small-latest
16:23:09.785 chat mistral-small-latest
16:23:09.789 chat mistral-small-latest
16:23:09.839 chat mistral-small-latest
16:23:09.868 chat mistral-small-latest
16:23:10.287 chat mistral-small-latest
16:23:10.297 chat mistral-small-latest
16:23:10.359 chat mistral-small-latest
16:23:10.456 chat mistral-small-latest
16:23:10.505 chat mistral-small-latest
16:23:10.923 chat mistral-small-latest
16:23:10.977 chat mistral

In [14]:
baseline = optimization.steps[0]
best = optimization.best

print(f"Baseline score: {baseline.score:.4f}")
print(f"Best score:     {best.score:.4f} (step {optimization.steps.index(best)})")
print()
print("Best prompt template:")
print(best.prompt_template)

Baseline score: 0.6667
Best score:     1.0000 (step 3)

Best prompt template:
You must analyze the following product review and output the sentiment and the reason.

CRITICAL: The "reason" field MUST be written in the EXACT same language as the review.
- Review is in English -> "reason" MUST be in English.
- Review is in Spanish -> "reason" MUST be in Spanish.
- Review is in German -> "reason" MUST be in German.
- Review is in Italian -> "reason" MUST be in Italian.
- Review is in any other language -> "reason" MUST be in that exact same language.

Product Review:
"""
{{ REVIEW }}
"""

Determine if the sentiment is 'positive', 'neutral', or 'negative'. 
Note: If the reviewer is satisfied, has no complaints, or says the product works fine, classify the sentiment as 'positive'.

Write the "reason" in the same language as the review.


This is the complete trajectory:

In [15]:
from promptuna.report import render_history

history = render_history(steps=optimization.steps)
display(Markdown(history))

## Step 0 — baseline · score 0.67

### Template



````template
Your task is to classify the sentiment of this product review:

{{ REVIEW }}

The possible labels are 'positive', 'neutral' and 'negative'.

Write the reason in the same language as the review.

````

### Quality

| Metric | Score | Replicate noise |
| --- | --- | --- |
| `label_correctness` | 0.96 ± 0.19 (n=27) | 0.00 ± 0.00 (n=27) |
| `reason_language` | 0.37 ± 0.49 (n=27) | 0.00 ± 0.00 (n=27) |

### Reliability

- **Trials**: 27 successful / 27 total
- **Trial failure rate**: 0.00%
- **Scorings**: 54 successful / 54 total
- **Scoring failure rate**: 0.00%

---

## Step 1 — candidate · score 0.61 · Δ -0.06 vs baseline

### Intent



**Hypothesis:** If we explicitly instruct the model to first identify the language of the review, and then strictly mandate that the 'reason' field must be written in that exact identified language, the reason_language score will improve significantly without hurting label_correctness.



**Edit plan:** We will refine the baseline template. We will add a highly visible, structured instruction block that forces the model to: 1. Identify the language of the review. 2. Draft the reason in that identified language. 3. Populate the 'reason' field using that draft. We will use strong, imperative language and clear formatting.

### Template



````template
Your task is to classify the sentiment of this product review and provide a short justification.

CRITICAL REQUIREMENT:
You must write the "reason" field in the EXACT same language as the product review itself. 
- If the review is in English, the reason must be in English.
- If the review is in Spanish, the reason must be in Spanish.
- If the review is in German, the reason must be in German.
(And so on for any language used in the review).

Here is the product review:
"""
{{ REVIEW }}
"""

The possible sentiment labels are: 'positive', 'neutral', 'negative'.

Step-by-step instructions:
1. Read the product review and identify its language.
2. Determine the sentiment ('positive', 'neutral', or 'negative').
3. Write a one-sentence justification ('reason') for this sentiment, ensuring you write it in the exact language you identified in Step 1.
````

### Quality

| Metric | Score | Replicate noise |
| --- | --- | --- |
| `label_correctness` | 0.96 ± 0.19 (n=27) | 0.00 ± 0.00 (n=27) |
| `reason_language` | 0.26 ± 0.45 (n=27) | 0.00 ± 0.00 (n=27) |

### Reliability

- **Trials**: 27 successful / 27 total
- **Trial failure rate**: 0.00%
- **Scorings**: 54 successful / 54 total
- **Scoring failure rate**: 0.00%

---

## Step 2 — candidate · score 0.96 · Δ +0.30 vs baseline

### Intent



**Hypothesis:** If we place the language constraint directly inside the definition of the task and use extremely concise, high-impact framing (potentially matching the language of the instruction to the expected behavior, or using a highly structured format that pairs the review directly with the output language), the model will respect the language constraint. Let's try a extremely direct, zero-fluff prompt that explicitly forbids Spanish unless the review itself is in Spanish.



**Edit plan:** We will discard the verbose Step 1 template and return to a highly streamlined template. We will explicitly state: 'If the review is in English, you MUST write the reason in English. If the review is in Spanish, you MUST write the reason in Spanish.' We will place this instruction at the very beginning and the very end of the prompt to maximize attention, keeping it extremely brief to avoid structured-output distraction.

### Template



````template
You must analyze the following product review and output the sentiment and the reason.

CRITICAL: The "reason" field MUST be written in the EXACT same language as the review.
- Review is in English -> "reason" MUST be in English.
- Review is in Spanish -> "reason" MUST be in Spanish.
- Review is in German -> "reason" MUST be in German.

Product Review:
"""
{{ REVIEW }}
"""

Determine if the sentiment is 'positive', 'neutral', or 'negative'. Write the "reason" in the same language as the review.
````

### Quality

| Metric | Score | Replicate noise |
| --- | --- | --- |
| `label_correctness` | 0.96 ± 0.19 (n=27) | 0.00 ± 0.00 (n=27) |
| `reason_language` | 0.96 ± 0.19 (n=27) | 0.00 ± 0.00 (n=27) |

### Reliability

- **Trials**: 27 successful / 27 total
- **Trial failure rate**: 0.00%
- **Scorings**: 54 successful / 54 total
- **Scoring failure rate**: 0.00%

---

## Step 3 — candidate · score 1.00 · Δ +0.33 vs baseline · ⭐ best

### Intent



**Hypothesis:** If we generalize the language mapping rule to explicitly include Italian (and other common languages) while keeping the prompt extremely concise, and add a subtle hint to classify mildly positive/satisfied reviews ('nothing to complain about', 'works fine') as 'positive' rather than 'neutral', we will fix both remaining failure modes and achieve a perfect score.



**Edit plan:** We will refine the highly successful Step 2 template. We will: 1. Expand the language mapping list to explicitly include Italian (since it appeared in the failures) and a general fallback rule. 2. Add a brief, clear instruction clarifying that reviews expressing satisfaction or 'no complaints' should be classified as 'positive' rather than 'neutral'.

### Template



````template
You must analyze the following product review and output the sentiment and the reason.

CRITICAL: The "reason" field MUST be written in the EXACT same language as the review.
- Review is in English -> "reason" MUST be in English.
- Review is in Spanish -> "reason" MUST be in Spanish.
- Review is in German -> "reason" MUST be in German.
- Review is in Italian -> "reason" MUST be in Italian.
- Review is in any other language -> "reason" MUST be in that exact same language.

Product Review:
"""
{{ REVIEW }}
"""

Determine if the sentiment is 'positive', 'neutral', or 'negative'. 
Note: If the reviewer is satisfied, has no complaints, or says the product works fine, classify the sentiment as 'positive'.

Write the "reason" in the same language as the review.
````

### Quality

| Metric | Score | Replicate noise |
| --- | --- | --- |
| `label_correctness` | 1.00 ± 0.00 (n=27) | 0.00 ± 0.00 (n=27) |
| `reason_language` | 1.00 ± 0.00 (n=27) | 0.00 ± 0.00 (n=27) |

### Reliability

- **Trials**: 27 successful / 27 total
- **Trial failure rate**: 0.00%
- **Scorings**: 54 successful / 54 total
- **Scoring failure rate**: 0.00%

### Error Analysis

All examples scored perfectly.